<a href="https://colab.research.google.com/github/bimal66215/Assignments/blob/main/Feature%20extraction%20and%20LORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Different Types of Fine-tuning for Custom Classification Tasks

Fine-tuning is a powerful technique for adapting a pre-trained model to a specific downstream task with a smaller, task-specific dataset. Here are several common approaches:

### 1. Full Fine-tuning (Standard Fine-tuning)

*   **Description**: This is the most straightforward approach where all layers of the pre-trained model are unfrozen and updated during training on the new dataset. The learning rate is typically much smaller than the pre-training learning rate.
*   **When to Use**: When you have a moderate to large amount of task-specific data and want to achieve the highest possible performance, or when your new task is significantly different from the pre-training task.
*   **Pros**: Can achieve state-of-the-art performance.
*   **Cons**: Computationally expensive, requires more data to prevent overfitting, and can be slower to train.

### 2. Feature Extraction (Linear Probing/Classifier-only Fine-tuning)

*   **Description**: The pre-trained model's convolutional or transformer layers are kept frozen, and only the final classification layer(s) are trained on the new dataset. The frozen layers act as a fixed feature extractor.
*   **When to Use**: When you have very limited task-specific data, or when the new task is very similar to the pre-training task.
*   **Pros**: Computationally efficient, less prone to overfitting with small datasets, faster training.
*   **Cons**: Performance might be lower than full fine-tuning if the pre-trained features aren't perfectly aligned with the new task.

### 3. Layer-wise (or Discriminative) Fine-tuning

*   **Description**: This approach involves setting different learning rates for different layers of the model. Typically, earlier layers (which capture more general features) are trained with very small learning rates, while later layers (which capture more task-specific features) are trained with larger learning rates.
*   **When to Use**: A good balance between full fine-tuning and feature extraction, especially when you have a decent amount of data but want to maintain the general knowledge learned by earlier layers.
*   **Pros**: Can adapt more effectively than feature extraction while being more stable than full fine-tuning, especially with varied data sizes.
*   **Cons**: Requires careful tuning of layer-wise learning rates.

### 4. Adapter-based Fine-tuning

*   **Description**: Instead of modifying the entire pre-trained model, small, task-specific neural network modules (adapters) are inserted between the layers of the pre-trained model. Only these adapter modules are trained, keeping the original pre-trained weights frozen.
*   **When to Use**: When you want to fine-tune a single model for many different tasks without storing a full copy of the model for each task (parameter efficient), or when memory is a concern.
*   **Pros**: Highly parameter-efficient (stores only a small fraction of parameters per task), faster training, less prone to catastrophic forgetting.
*   **Cons**: Might add a slight overhead to inference time, and adapter architecture design can be a hyperparameter.

### 5. Prompt-tuning / Prefix-tuning (for Large Language Models)

*   **Description**: Instead of fine-tuning model weights, these methods optimize a small, continuous vector (the "prompt" or "prefix") that is prepended to the input embeddings. The pre-trained model weights remain frozen. The model then conditions its generation or classification on this learned prompt.
*   **When to Use**: Primarily for large language models (LLMs) when you want to adapt them to new tasks without modifying their huge number of parameters, making it very memory and computationally efficient.
*   **Pros**: Extremely parameter-efficient, avoids catastrophic forgetting, faster to train than full fine-tuning for LLMs, and can be used for multi-tasking by learning different prompts for different tasks.
*   **Cons**: Can be less effective than full fine-tuning for tasks that require deep architectural changes, and designing effective prompts can be challenging.

### 6. LoRA (Low-Rank Adaptation) / QLoRA

*   **Description**: LoRA inserts small, trainable matrices into the pre-trained model's layers. Instead of training the full weight matrices, only these low-rank decomposition matrices are trained. QLoRA extends LoRA by quantizing the base model weights to 4-bit and then training LoRA adapters on top of these quantized weights.
*   **When to Use**: When working with very large models (like diffusion models or large language models) and memory/computation are major constraints, but you still want performance close to full fine-tuning.
*   **Pros**: Highly parameter-efficient, significant memory savings (especially QLoRA), faster training, and maintains strong performance.
*   **Cons**: Can add some complexity to the model architecture and deployment.

### How to choose?

The best fine-tuning method depends on your specific task, the size of your dataset, and available computational resources:

*   **Small dataset, similar task**: Feature Extraction is a good start.
*   **Small to medium dataset, somewhat different task**: Layer-wise Fine-tuning or Adapter-based methods.
*   **Large dataset, very different task**: Full Fine-tuning.
*   **Very large pre-trained model (especially LLMs/Diffusion models), resource-constrained**: Prompt-tuning, Prefix-tuning, LoRA, or QLoRA.

## LoRA Fine-tuning Example for Custom Classification (PyTorch)

First, let's install the necessary libraries: `transformers` for the pre-trained model and tokenizer, `peft` for easily integrating LoRA, and `accelerate` (a common dependency for `transformers` training).

In [2]:
import sys
!{sys.executable} -m pip uninstall -y torchao # Uninstall existing torchao
!{sys.executable} -m pip install --force-reinstall --no-cache-dir torchao>=0.16.0
!{sys.executable} -m pip install transformers peft accelerate torch datasets

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Next, we'll import the required libraries and set up the device for training (GPU if available, otherwise CPU).

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


We'll choose a small pre-trained model, like `bert-base-uncased`, and its corresponding tokenizer. For classification, we'll initialize it with 2 labels (for a binary classification task).

In [4]:
model_name = "bert-base-uncased"
num_labels = 6 # The emotion dataset has 6 labels

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


To simulate a 'custom classification task', let's create a simple synthetic dataset. This dataset will contain example texts and corresponding labels (0 or 1).

In [5]:
from datasets import load_dataset

# Load the 'emotion' dataset with its full path
raw_datasets = load_dataset('dair-ai/emotion')

# Define a tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Apply the tokenization function to the dataset
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Rename the 'label' column to 'labels' to match what Trainer expects
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Format the datasets for PyTorch
tokenized_datasets.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

# Extract training and validation sets
train_dataset = tokenized_datasets["train"]
val_dataset = tokenized_datasets["validation"]

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

# Display the first example from the training set
print("\nFirst training example:")
print(train_dataset[0])

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training examples: 16000
Validation examples: 2000

First training example:
{'labels': tensor(0), 'input_ids': tensor([  101,  1045,  2134,  2102,  2514, 26608,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,   

Now, we configure LoRA using `LoraConfig` from the `peft` library. We specify the `task_type` as sequence classification, `r` (LoRA rank), `lora_alpha` (scaling factor), and `lora_dropout`.

Then, we use `get_peft_model` to wrap our pre-trained model with the LoRA adapters. This will make only the LoRA parameters trainable, significantly reducing the number of parameters to update.

In [6]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,  # LoRA rank
    lora_alpha=32, # LoRA scaling factor
    lora_dropout=0.1,
    # target_modules=["query", "value"] # Specify which modules to apply LoRA to, e.g., attention weights
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 299,526 || all params: 109,786,380 || trainable%: 0.2728


We'll define the training arguments using `TrainingArguments` and set up the `Trainer` from `transformers`. We'll also define a `compute_metrics` function for evaluation during training.

In [8]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir="./lora_results",
    learning_rate=2e-4, # LoRA often uses slightly higher learning rates
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3, # Small number of epochs for demonstration
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none", # Prevents logging to W&B etc. if not configured
)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics,
# )

# print("Starting LoRA fine-tuning...")
# trainer.train()
# print("Fine-tuning complete.")

Finally, let's evaluate the fine-tuned LoRA model on the validation set.

In [ ]:
print("Evaluating LoRA fine-tuned model...")
results = trainer.evaluate()
print(results)

In [ ]:
output_dir = "./lora_finetuned_model"
model.save_pretrained(output_dir)
print(f"LoRA adapters saved to {output_dir}")

LoRA adapters saved to ./lora_finetuned_model


To load the fine-tuned model later for inference, you would typically do something like this:

```python
from transformers import AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig

# Load the base model
base_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

# Load the LoRA configuration and then the PeftModel
config = PeftConfig.from_pretrained(output_dir)
model_to_load = PeftModel.from_pretrained(base_model, output_dir)

# You can also merge the adapters into the base model if you want a single, consolidated model
# model_to_load = model_to_load.merge_and_unload()

print("Fine-tuned model loaded successfully!")
```

### Test Prediction with Saved LoRA Model

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig

# Define the base model name and number of labels (from previous steps)
model_name = "bert-base-uncased"
num_labels = 6 # As determined by the 'emotion' dataset

# Load the tokenizer (ensure it's the same one used for training)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the base model without the classification head (it will be added by PeftModel)
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Load the LoRA configuration and then the PeftModel
output_dir = "./lora_finetuned_model" # Directory where LoRA adapters were saved
peft_config = PeftConfig.from_pretrained(output_dir)
loaded_peft_model = PeftModel.from_pretrained(base_model, output_dir)

# Set the model to evaluation mode
loaded_peft_model.eval()
loaded_peft_model.to(device)

print("Fine-tuned LoRA model and tokenizer loaded successfully for inference.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fine-tuned LoRA model and tokenizer loaded successfully for inference.


In [ ]:
# Example text for prediction
test_text = "I am feeling very happy today!"

# Tokenize the input text
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Perform inference
with torch.no_grad():
    outputs = loaded_peft_model(**inputs)
    logits = outputs.logits

# Convert logits to probabilities using softmax
probabilities = torch.softmax(logits, dim=-1)

# Get the predicted class (index with highest probability)
predicted_class_id = torch.argmax(probabilities, dim=-1).item()
predicted_probability = probabilities[0, predicted_class_id].item()

# Map the predicted class ID back to an emotion label
# The 'emotion' dataset labels are: 0: sadness, 1: joy, 2: love, 3: anger, 4: fear, 5: surprise
label_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
predicted_label = label_map.get(predicted_class_id, f"Unknown label ID: {predicted_class_id}")

print(f"Test text: {test_text}")
print(f"Predicted emotion ID: {predicted_class_id}")
print(f"Predicted emotion: {predicted_label} (Probability: {predicted_probability:.4f})")

# Get top 5 most probable classes
top5_probabilities, top5_indices = torch.topk(probabilities, k=5, dim=-1)

print("\nTop 5 Most Probable Emotions:")
for i in range(top5_probabilities.shape[1]):
    label_id = top5_indices[0, i].item()
    probability = top5_probabilities[0, i].item()
    label_name = label_map.get(label_id, f"Unknown label ID: {label_id}")
    print(f"- {label_name}: {probability:.4f}")

Test text: I am feeling very happy today!
Predicted emotion ID: 1
Predicted emotion: joy (Probability: 0.9999)

Top 5 Most Probable Emotions:
- joy: 0.9999
- love: 0.0000
- sadness: 0.0000
- anger: 0.0000
- surprise: 0.0000


The model has now made a prediction on the sample text, classifying it into one of the `emotion` categories based on the fine-tuning it received.

### Prediction Function

In [ ]:
def predict_emotion(sentence: str, confidence_threshold: float = 0.7):
    """
    Predicts the emotion of a given sentence using the loaded LoRA fine-tuned model,
    with an option for a confidence threshold.

    Args:
        sentence (str): The input text sentence.
        confidence_threshold (float): The minimum probability for a prediction to be
                                      considered certain. If the highest probability
                                      is below this, 'uncertain' is returned.

    Returns:
        dict: A dictionary containing the predicted label, its probability,
              and a list of the top 5 most probable emotions with their probabilities.
    """

    # Ensure label_map is accessible (from the previous cell)
    label_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

    # Tokenize the input text
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Perform inference
    with torch.no_grad():
        outputs = loaded_peft_model(**inputs)
        logits = outputs.logits

    # Convert logits to probabilities using softmax
    probabilities = torch.softmax(logits, dim=-1)

    # Get the predicted class (index with highest probability)
    predicted_class_id = torch.argmax(probabilities, dim=-1).item()
    predicted_probability = probabilities[0, predicted_class_id].item()
    predicted_label = label_map.get(predicted_class_id, f"Unknown label ID: {predicted_class_id}")

    # Apply confidence threshold
    if predicted_probability < confidence_threshold:
        predicted_label = "uncertain"

    # Get top 5 most probable classes
    top5_probabilities, top5_indices = torch.topk(probabilities, k=5, dim=-1)
    top5_results = []
    for i in range(top5_probabilities.shape[1]):
        label_id = top5_indices[0, i].item()
        prob = top5_probabilities[0, i].item()
        label_name = label_map.get(label_id, f"Unknown label ID: {label_id}")
        top5_results.append({'label': label_name, 'probability': round(prob, 5)})

    return {
        'predicted_label': predicted_label,
        'predicted_probability': predicted_probability, # Keep actual probability even if 'uncertain'
        'top5_emotions': top5_results
    }

# Demo of the function with threshold
example_sentence = "This is absolutely terrible, I am so angry!"
prediction_results = predict_emotion(example_sentence, confidence_threshold=0.9)

print(f"\nSentence: '{example_sentence}'")
print(f"Predicted Emotion: {prediction_results['predicted_label']} (Raw Probability: {prediction_results['predicted_probability']:.4f})")
print("Top 5 Most Probable Emotions:")
for item in prediction_results['top5_emotions']:
    print(f"- {item['label']}: {item['probability']:.4f}")

example_sentence_2 = "I woke up feeling wonderful and excited for the day!"
prediction_results_2 = predict_emotion(example_sentence_2, confidence_threshold=0.9)

print(f"\nSentence: '{example_sentence_2}'")
print(f"Predicted Emotion: {prediction_results_2['predicted_label']} (Raw Probability: {prediction_results_2['predicted_probability']:.4f})")
print("Top 5 Most Probable Emotions:")
for item in prediction_results_2['top5_emotions']:
    print(f"- {item['label']}: {item['probability']:.4f}")

# Test with your example sentence and a threshold
user_example_sentence = "Help me purchase a Product"
user_prediction_results = predict_emotion(user_example_sentence, confidence_threshold=0.9)

print(f"\nSentence: '{user_example_sentence}' (with threshold 0.9)")
print(f"Predicted Emotion: {user_prediction_results['predicted_label']} (Raw Probability: {user_prediction_results['predicted_probability']:.4f})")
print("Top 5 Most Probable Emotions:")
for item in user_prediction_results['top5_emotions']:
    print(f"- {item['label']}: {item['probability']:.4f}")


Sentence: 'This is absolutely terrible, I am so angry!'
Predicted Emotion: anger (Raw Probability: 0.9997)
Top 5 Most Probable Emotions:
- anger: 0.9997
- fear: 0.0002
- sadness: 0.0001
- love: 0.0000
- joy: 0.0000

Sentence: 'I woke up feeling wonderful and excited for the day!'
Predicted Emotion: joy (Raw Probability: 0.9998)
Top 5 Most Probable Emotions:
- joy: 0.9998
- surprise: 0.0001
- love: 0.0001
- sadness: 0.0000
- anger: 0.0000

Sentence: 'Help me purchase a Product' (with threshold 0.9)
Predicted Emotion: joy (Raw Probability: 0.9789)
Top 5 Most Probable Emotions:
- joy: 0.9789
- anger: 0.0122
- sadness: 0.0037
- love: 0.0026
- fear: 0.0023


In [ ]:
predict_emotion("Help me purchase a Product")

{'predicted_label': 'joy',
 'predicted_probability': 0.9789395332336426,
 'top5_emotions': [{'label': 'joy', 'probability': 0.97894},
  {'label': 'anger', 'probability': 0.01217},
  {'label': 'sadness', 'probability': 0.00368},
  {'label': 'love', 'probability': 0.00263},
  {'label': 'fear', 'probability': 0.00232}]}

## Feature Extraction for Custom Classification (PyTorch)

In feature extraction, we leverage the pre-trained model's layers to extract meaningful features from the input text, but we freeze these layers. Only a newly added classification head (or the existing one, if applicable) is trained on the task-specific data. This is efficient as fewer parameters are updated.

Let's set up the model for feature extraction.

In [9]:
from transformers import AutoModelForSequenceClassification

# Reload the pre-trained model for feature extraction
# This time, we don't apply LoRA config.
model_fe = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Freeze the base model's parameters
for param in model_fe.base_model.parameters():
    param.requires_grad = False

# The classification head will be trained by default
print("Model for Feature Extraction:")
print(model_fe)

# Verify which parameters are trainable
trainable_params = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model_fe.parameters())
print(f"\nTrainable parameters for Feature Extraction: {trainable_params}")
print(f"Total parameters for Feature Extraction: {all_params}")
print(f"Trainable %: {100 * trainable_params / all_params:.4f}%")

model_fe.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model for Feature Extraction:
BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Now we'll define `TrainingArguments` and set up the `Trainer` for our feature extraction model. Note that the dataset (`train_dataset`, `val_dataset`) and the `compute_metrics` function from the LoRA example can be reused as they are task-agnostic.

In [10]:
training_args_fe = TrainingArguments(
    output_dir="./fe_results",
    learning_rate=5e-5, # Typically a smaller learning rate than full fine-tuning, but can be larger than LoRA's if only head is trained
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5, # More epochs might be needed as only the head is learning
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer_fe = Trainer(
    model=model_fe,
    args=training_args_fe,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting Feature Extraction training...")
trainer_fe.train()
print("Feature Extraction training complete.")

Starting Feature Extraction training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.570592,1.555903,0.395500,0.288856,0.249349,0.395500
2,1.546435,1.539483,0.403500,0.299967,0.252123,0.403500
3,1.542031,1.534236,0.427000,0.319712,0.270358,0.427000
4,1.513676,1.526982,0.433000,0.331348,0.269717,0.433000
5,1.532516,1.524214,0.434000,0.327266,0.272659,0.434000


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Feature Extraction training complete.


Finally, let's evaluate the feature extraction model and save it.

In [11]:
print("Evaluating Feature Extraction model...")
results_fe = trainer_fe.evaluate()
print(results_fe)

output_dir_fe = "./fe_finetuned_model"
trainer_fe.save_model(output_dir_fe)
print(f"Feature extraction model saved to {output_dir_fe}")

Evaluating Feature Extraction model...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
1.532516,1.526982,5,0.433000,0.331348,0.269717,0.433000


{'eval_loss': 1.5269817113876343, 'eval_accuracy': 0.433, 'eval_f1': 0.3313477274965637, 'eval_precision': 0.2697174695469846, 'eval_recall': 0.433}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Feature extraction model saved to ./fe_finetuned_model


### Loading and Evaluating the Existing LoRA Fine-tuned Model

Since the LoRA adapters (`adapter_config.json` and `adapter_model.safetensors`) are already present in `/content/`, we will load the base model and then apply these adapters to it. After loading, we will evaluate this LoRA model to get its performance metrics.

In [12]:
from peft import PeftModel, PeftConfig

# Define the base model name and number of labels
# These are already defined in previous cells: model_name, num_labels, device

# Load the base model (a fresh instance) to attach LoRA adapters to
base_model_for_lora_eval = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

# The adapter config and model are in '/content/' directly
lora_adapter_path = "/content/"

# Load the adapters onto the base model
loaded_lora_model = PeftModel.from_pretrained(base_model_for_lora_eval, lora_adapter_path)

loaded_lora_model.eval() # Set model to evaluation mode
print("LoRA model loaded successfully for evaluation.")

# Reuse training_args from the previously defined LoRA configuration (`a5b7796c`)
# If `training_args` is not directly available, it should be re-defined or imported from `lora_results`
# For evaluation, we mainly need the eval_dataset and compute_metrics

trainer_lora_eval = Trainer(
    model=loaded_lora_model,
    args=TrainingArguments(output_dir="./lora_eval_results", remove_unused_columns=False), # Minimal args for evaluation
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Evaluating LoRA fine-tuned model...")
results_lora = trainer_lora_eval.evaluate()
print(results_lora)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_re

LoRA model loaded successfully for evaluation.
Evaluating LoRA fine-tuned model...


Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
No log,0.216283,0,0.932500,0.932385,0.933423,0.932500


{'eval_loss': 0.21628320217132568, 'eval_accuracy': 0.9325, 'eval_f1': 0.9323852835933799, 'eval_precision': 0.9334228847383881, 'eval_recall': 0.9325}


### Comparison of Feature Extraction vs. LoRA Fine-tuning

Let's compare the evaluation metrics from both approaches:

In [13]:
print("\n--- Comparison of Results ---")
print("\nFeature Extraction Model Results:")
for metric, value in results_fe.items():
    print(f"  {metric}: {value:.4f}")

print("\nLoRA Fine-tuned Model Results:")
for metric, value in results_lora.items():
    print(f"  {metric}: {value:.4f}")

# You can also perform a more direct comparison for key metrics
print("\n--- Key Metric Comparison ---")
print(f"F1 Score (Feature Extraction): {results_fe['eval_f1']:.4f}")
print(f"F1 Score (LoRA):               {results_lora['eval_f1']:.4f}")
print(f"Accuracy (Feature Extraction): {results_fe['eval_accuracy']:.4f}")
print(f"Accuracy (LoRA):               {results_lora['eval_accuracy']:.4f}")

# Compare trainable parameters
lora_trainable_params_count = sum(p.numel() for p in loaded_lora_model.parameters() if p.requires_grad)
# fe_trainable_params_count is available from previous cell execution as `trainable_params`
print(f"\nTrainable Parameters (Feature Extraction): {trainable_params}")
print(f"Trainable Parameters (LoRA):               {lora_trainable_params_count}")


--- Comparison of Results ---

Feature Extraction Model Results:
  eval_loss: 1.5270
  eval_accuracy: 0.4330
  eval_f1: 0.3313
  eval_precision: 0.2697
  eval_recall: 0.4330

LoRA Fine-tuned Model Results:
  eval_loss: 0.2163
  eval_accuracy: 0.9325
  eval_f1: 0.9324
  eval_precision: 0.9334
  eval_recall: 0.9325

--- Key Metric Comparison ---
F1 Score (Feature Extraction): 0.3313
F1 Score (LoRA):               0.9324
Accuracy (Feature Extraction): 0.4330
Accuracy (LoRA):               0.9325

Trainable Parameters (Feature Extraction): 4614
Trainable Parameters (LoRA):               0
